# Caso G · 01 Reglas de calidad sobre la capa bronce

> _Tutorial · Caso de uso: **G — Calidad con agentes** · Capa Medallion: **bronce** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Definir y ejecutar reglas Great-Expectations-style sobre los CSV originales (In-Gauge, BDG2, LBNL FDD) **sin** depender de Influx levantado.


## 2. Qué se aprende

- Por qué la calidad bronce es la primera línea de defensa.
- Reglas: rangos, no nulos, tipos.
- Cómo escribir reglas sin Great Expectations (`pandera`-lite).


## 3. Contexto del caso de uso

Equipo G empieza semana 1: reglas bronce sobre CSV de los demás equipos. Estrategia anti-bloqueo.


## 4. Relación con CENTINELA+

Las reglas bronce protegen el ETL real desde el día 1.


## 5. Relación con Medallion

Bronce.


## 6. Datos de entrada

Mocks de los demás casos.


## 7. Schema CAPTIA esperado

No aplica (aún en bronce).


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos los 3 CSV.


In [ ]:
ingauge = pd.read_csv(ROOT / "notebooks/_data/ingauge_aula01_mock.csv", comment="#", parse_dates=["timestamp"])
bdg2 = pd.read_csv(ROOT / "notebooks/_data/bdg2_education_subset_mock.csv", comment="#", parse_dates=["timestamp"])
lbnl = pd.read_csv(ROOT / "notebooks/_data/lbnl_fdd_rtu_mock.csv", comment="#", parse_dates=["timestamp"])
print({"ingauge": ingauge.shape, "bdg2": bdg2.shape, "lbnl": lbnl.shape})


## 10. Exploración paso a paso

Definimos un mini DSL de reglas.


In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Rule:
    name: str
    fn: Callable[[pd.DataFrame], bool]
    description: str

rules = [
    Rule("ingauge_co2_range",
         lambda d: d["Indoor_CO2"].between(300, 5000).all(),
         "CO2 entre 300 y 5000 ppm"),
    Rule("ingauge_no_negative_people",
         lambda d: (d["People_Count"] >= 0).all(),
         "people_count no negativo"),
    Rule("bdg2_power_nonneg",
         lambda d: (d["power_kw"] >= 0).all(),
         "power no negativo"),
    Rule("lbnl_dt_supply_return",
         lambda d: ((d["RA_TEMP"] - d["SA_TEMP"]) >= -2).all(),
         "ΔT supply-return físicamente plausible"),
]
print(f"{len(rules)} reglas registradas")


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Reporte por dataset.


In [ ]:
def evaluate(rules, ds_map):
    rows = []
    for r in rules:
        for ds_name, ds in ds_map.items():
            try:
                ok = bool(r.fn(ds))
            except KeyError:
                continue
            rows.append({"rule": r.name, "dataset": ds_name, "ok": ok})
    return pd.DataFrame(rows)

report = evaluate(rules, {"ingauge": ingauge, "bdg2": bdg2, "lbnl": lbnl})
report


## 13. Visualizaciones explicativas

Heatmap.


In [ ]:
heat = report.pivot(index="rule", columns="dataset", values="ok").fillna("-")
heat.applymap(lambda v: "✓" if v is True else ("✗" if v is False else "—"))


## 14. Validaciones

Todo OK.


In [ ]:
fails = report[report["ok"] == False]
assert fails.empty, f"Reglas falladas: {fails}"
print("Bronze quality: PASS")


## 15. Errores comunes

1. Reglas demasiado estrictas que rechazan datos reales.
2. Reglas que tardan demasiado en grandes datasets.
3. No diferenciar warning de error.


## 16. Ejercicios propuestos

1. Añade una regla de monotonicidad temporal.
2. Convierte las reglas a Great Expectations.
3. Diseña una regla para detectar outliers > 3σ.


## 17. Cómo se reutiliza con datos reales

Idéntico — solo cambia origen.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `07_case_G_data_quality_agents/02_reglas_calidad_plata_influxdb.ipynb`.
- Documento web del caso: `docs/validation/data-quality.md`.
